# Inspect BM25 Maintenance: Vocabulary Drift Analysis

This notebook analyzes the BM25 vocabulary stored in BigQuery and evaluates whether a rebuild is needed. It computes drift metrics by comparing the current corpus against the trained vocabulary:

- **OOV rate** — % of terms in current corpus not in the stored vocabulary
- **Stale rate** — % of vocabulary terms no longer in the current corpus
- **Corpus size delta** — % change in total chunks since last training

If any metric exceeds its threshold (configurable in `config.py`), a rebuild is recommended. Use `refresh_bm25.py` (step 9) for automated drift detection and conditional retrain, then `apply_bm25.py` (step 7b) to compute new sparse embeddings.

---
## Setup

In [ ]:
import string
from collections import Counter

import nltk
import pandas as pd
from google.cloud import bigquery
from config import (
    BQ_PROJECT, BQ_DATASET, BQ_TABLE_PREFIX,
    BM25_NGRAMS,
    BM25_OOV_THRESHOLD, BM25_STALE_THRESHOLD, BM25_CORPUS_DELTA,
)

nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = nltk.stem.WordNetLemmatizer()
stop_words = set(nltk.corpus.stopwords.words('english'))

bq = bigquery.Client(project=BQ_PROJECT)


def chunk_prep(text, ngrams=BM25_NGRAMS):
    """Preprocess text for BM25."""
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.tokenize.word_tokenize(text)
    unigrams = [lemmatizer.lemmatize(t) for t in tokens
                if len(t) > 1 and t not in stop_words]
    if ngrams >= 2:
        all_grams = unigrams.copy()
        for n in range(2, ngrams + 1):
            all_grams.extend(' '.join(g) for g in zip(*[unigrams[i:] for i in range(n)]))
        return all_grams
    return unigrams


def build_bm25_text(row, source_type):
    """Construct BM25 input: text_content + enrichment tags."""
    parts = [row.text_content]
    if row.topics:
        parts.extend(row.topics)
    if source_type == 'reddit' and hasattr(row, 'methods_mentioned') and row.methods_mentioned:
        parts.extend(row.methods_mentioned)
    elif source_type == 'zoom' and hasattr(row, 'action_items') and row.action_items:
        parts.extend(row.action_items)
    elif source_type == 'pdf' and hasattr(row, 'functions_referenced') and row.functions_referenced:
        parts.extend(row.functions_referenced)
    return ' '.join(parts)


print(f'Thresholds: OOV={BM25_OOV_THRESHOLD:.0%}, Stale={BM25_STALE_THRESHOLD:.0%}, Corpus delta={BM25_CORPUS_DELTA:.0%}')

---
## Load Current Model Metadata

In [ ]:
model_table = f'{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_bm25_model'
model_meta = list(bq.query(f'SELECT * FROM `{model_table}` ORDER BY model_version DESC LIMIT 1').result())[0]

print(f'Model version: {model_meta.model_version}')
print(f'Trained at: {model_meta.trained_at}')
print(f'Corpus size: {model_meta.corpus_size} chunks')
print(f'Vocabulary size: {model_meta.vocab_size} terms')
print(f'Average document length: {model_meta.avgdl:.1f}')
print(f'Parameters: k1={model_meta.k1}, b={model_meta.b}, epsilon={model_meta.epsilon}, ngrams={model_meta.ngrams}')

---
## Load Stored Vocabulary

In [ ]:
vocab_table = f'{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_bm25_vocabulary'
vocab_rows = list(bq.query(
    f'SELECT term, idf FROM `{vocab_table}` WHERE model_version = {model_meta.model_version}'
).result())
stored_vocab = {row.term for row in vocab_rows}

print(f'Stored vocabulary (version {model_meta.model_version}): {len(stored_vocab)} terms')

# Top IDF terms
idf_sorted = sorted(vocab_rows, key=lambda r: r.idf, reverse=True)
print(f'\nTop 10 IDF terms (most discriminating):')
for row in idf_sorted[:10]:
    print(f'  {row.idf:.4f}  {row.term}')

---
## Read Current Chunks & Compute New Corpus Stats

In [ ]:
sources = [
    ('reddit', 'SELECT chunk_id, text_content, topics, methods_mentioned'),
    ('zoom', 'SELECT chunk_id, text_content, topics, action_items'),
    ('pdf', 'SELECT chunk_id, text_content, topics, functions_referenced'),
]

current_terms = Counter()
current_corpus_size = 0

for source_type, query in sources:
    table_ref = f'{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_{source_type}_chunks'
    rows = list(bq.query(f'{query} FROM `{table_ref}`').result())
    for row in rows:
        bm25_text = build_bm25_text(row, source_type)
        tokens = chunk_prep(bm25_text)
        current_terms.update(set(tokens))  # count unique terms per doc
    current_corpus_size += len(rows)
    print(f'  {source_type}: {len(rows)} chunks')

current_vocab = set(current_terms.keys())
print(f'\nCurrent corpus: {current_corpus_size} chunks')
print(f'Current unique terms: {len(current_vocab)}')

---
## Drift Metrics

In [ ]:
# OOV: terms in current corpus NOT in stored vocabulary
oov_terms = current_vocab - stored_vocab
oov_rate = len(oov_terms) / len(current_vocab) if current_vocab else 0

# Stale: terms in stored vocabulary NOT in current corpus
stale_terms = stored_vocab - current_vocab
stale_rate = len(stale_terms) / len(stored_vocab) if stored_vocab else 0

# Corpus size delta
corpus_delta = abs(current_corpus_size - model_meta.corpus_size) / model_meta.corpus_size if model_meta.corpus_size else 0

print(f'OOV rate:         {oov_rate:.1%} ({len(oov_terms)} new terms)  [threshold: {BM25_OOV_THRESHOLD:.0%}]')
print(f'Stale rate:       {stale_rate:.1%} ({len(stale_terms)} stale terms)  [threshold: {BM25_STALE_THRESHOLD:.0%}]')
print(f'Corpus delta:     {corpus_delta:.1%} ({model_meta.corpus_size} -> {current_corpus_size})  [threshold: {BM25_CORPUS_DELTA:.0%}]')

---
## Threshold Evaluation

In [ ]:
rebuild_needed = False
reasons = []

if oov_rate >= BM25_OOV_THRESHOLD:
    rebuild_needed = True
    reasons.append(f'OOV rate {oov_rate:.1%} >= {BM25_OOV_THRESHOLD:.0%}')

if stale_rate >= BM25_STALE_THRESHOLD:
    rebuild_needed = True
    reasons.append(f'Stale rate {stale_rate:.1%} >= {BM25_STALE_THRESHOLD:.0%}')

if corpus_delta >= BM25_CORPUS_DELTA:
    rebuild_needed = True
    reasons.append(f'Corpus delta {corpus_delta:.1%} >= {BM25_CORPUS_DELTA:.0%}')

if rebuild_needed:
    print('REBUILD RECOMMENDED')
    for r in reasons:
        print(f'  - {r}')
    print('\nRetrain:  uv run python build_bm25.py (after incrementing BM25_MODEL_VERSION in config.py)')
    print('Then apply: uv run python apply_bm25.py')
else:
    print('No rebuild needed — all metrics within thresholds.')

---
## Term Frequency Comparison

Compare the top terms in the current corpus vs the stored vocabulary. Shifts in top terms indicate evolving content.

In [ ]:
# Top 20 terms by document frequency in current corpus
top_current = current_terms.most_common(20)

# Top 20 terms by IDF in stored vocabulary (inverse — low IDF = common)
idf_map = {row.term: row.idf for row in vocab_rows}

rows = []
for term, doc_freq in top_current:
    stored_idf = idf_map.get(term, None)
    rows.append({
        'term': term,
        'current_doc_freq': doc_freq,
        'stored_idf': f'{stored_idf:.4f}' if stored_idf else 'NEW',
        'status': 'new' if term not in stored_vocab else 'existing',
    })

print('Top 20 terms by document frequency in current corpus:\n')
pd.DataFrame(rows)

In [ ]:
# Sample OOV terms (if any)
if oov_terms:
    sample = sorted(oov_terms)[:20]
    print(f'Sample OOV terms ({len(oov_terms)} total):')
    for t in sample:
        print(f'  {t}')
else:
    print('No OOV terms — vocabulary fully covers current corpus.')